# 03. Feature Engineering, Stratified Partitioning, and Linear Encoding Pipeline
**CSE437 Final Project: Disparity and Error Analysis in Linear Income Classification**  
*Texas 2023 ACS 1-Year PUMS Microdata*

---

### Purpose and Methodological Justifications
1. **Educational Credential Tiering:** Collapses 24 granular Census `SCHL` codes into 4 ordinal tiers (`Less_than_HS`, `HS_or_Some_College`, `Bachelors`, `Graduate_Plus`) to establish clear credential boundaries for demographic parity evaluation[cite: 8].
2. **Feature Selection & Parsimony:** Excludes weeks worked (`WKWN`)[cite: 8]. Because the universe is already filtered to civilian full-time workers actively at work (`ESR == 1`, `WKHP >= 35`), `WKWN` exhibits near-zero variance (over 91% working 50–52 weeks) and introduces redundant collinearity[cite: 3].
3. **Stratified 80/20 Partitioning:** Partitions the 30,000-sample cohort into training ($N = 24,000$) and testing ($N = 6,000$) sets using stratified sampling on `HIGH_EARNER` to guarantee identical 74/26 class proportions across both splits[cite: 8].
4. **Leakage-Free Transformation Pipeline:**
   - **`StandardScaler`:** Continuous predictors (`AGEP`, `WKHP`) are centered to zero mean and scaled to unit variance[cite: 8]. This prevents continuous variables from dominating the margin objective of linear classifiers over one-hot binary indicators[cite: 8]. The scaler is **fit strictly on `X_train`** and applied to `X_test` without refitting to prevent test-set data leakage[cite: 8].
   - **`OneHotEncoder(drop='first')`:** Categorical features (`SEX_LABEL`, `COW_GROUP`, `OCCP_GROUP`, `SCHL_TIER`, `MAR`) are dummy-encoded dropping the reference level to eliminate the dummy variable trap and exact multicollinearity under linear optimization[cite: 8].
5. **Metadata Preservation:** Extracts unencoded demographic and institutional identifiers (`SEX_LABEL`, `COW_GROUP`, `SCHL_TIER`, `AGEP`) paired with true labels into `test_metadata.csv` for downstream subgroup auditing and hypothesis testing[cite: 8].

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Portable path resolution relative to repository root
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DATA_DIR = REPO_ROOT / "data" / "processed"
INPUT_CLEANED_FILE = PROCESSED_DATA_DIR / "texas_cleaned_30k.csv"

# Fallback resolution
if not INPUT_CLEANED_FILE.exists():
    if (Path.cwd() / "texas_cleaned_30k.csv").exists():
        INPUT_CLEANED_FILE = Path.cwd() / "texas_cleaned_30k.csv"
    elif (REPO_ROOT / "texas_cleaned_30k.csv").exists():
        INPUT_CLEANED_FILE = REPO_ROOT / "texas_cleaned_30k.csv"
    else:
        raise FileNotFoundError(
            f"Could not locate 'texas_cleaned_30k.csv'. Looked in:\n"
            f" - {INPUT_CLEANED_FILE}\n"
            f" - {Path.cwd() / 'texas_cleaned_30k.csv'}\n"
            "Please ensure notebook 02 has been executed to generate the processed sample."
        )

PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"[STATUS] Project root: {REPO_ROOT.resolve()}")
print(f"[STATUS] Ingesting cleaned sample from: {INPUT_CLEANED_FILE.resolve()}")

df = pd.read_csv(INPUT_CLEANED_FILE)
print(f"Ingested Dataset Shape: {df.shape[0]:,} rows, {df.shape[1]} columns")

## 1. Education Tier Bucketing and Demographic Labeling
We map 24 granular Census educational attainment codes into 4 standard analytical tiers and decode integer sex indicators into descriptive category labels[cite: 8].

In [ ]:
def bucket_education(code: float) -> str:
    """
    Collapses 24 granular Census SCHL attainment levels into 4 distinct tiers:
      - Less_than_HS:       Codes 1-15  (No high school diploma / GED)
      - HS_or_Some_College: Codes 16-20 (Regular diploma, GED, some college, Associate's)
      - Bachelors:          Code 21     (4-year Bachelor's degree)
      - Graduate_Plus:      Codes 22-24 (Master's, Professional degree, Doctorate)
    """
    c = int(code)
    if c < 16:
        return "Less_than_HS"
    elif 16 <= c <= 20:
        return "HS_or_Some_College"
    elif c == 21:
        return "Bachelors"
    else:
        return "Graduate_Plus"

# Apply educational tier mapping
df["SCHL_TIER"] = df["SCHL"].apply(bucket_education)

# Map numeric Census SEX (1: Male, 2: Female) to explicit strings
df["SEX_LABEL"] = df["SEX"].map({1: "Male", 2: "Female"})

print("Educational Attainment Distribution (30k Sample):")
print(df["SCHL_TIER"].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))
print("\nDemographic Sex Breakdown (30k Sample):")
print(df["SEX_LABEL"].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))

## 2. Feature Selection & Target Isolation
We select the final 7 candidate predictors (5 categorical and 2 continuous)[cite: 8]. Weeks worked (`WKWN`) is intentionally omitted due to near-zero variance within this full-time employed universe[cite: 3].

In [ ]:
# Define predictive feature subsets
categorical_features = ["SEX_LABEL", "COW_GROUP", "OCCP_GROUP", "SCHL_TIER", "MAR"]
numeric_features = ["AGEP", "WKHP"]
feature_cols = categorical_features + numeric_features

X = df[feature_cols].copy()
y = df["HIGH_EARNER"].copy()

print("=" * 65)
print("FEATURE SELECTION SUMMARY")
print("=" * 65)
print(f"Categorical Predictors ({len(categorical_features)}): {categorical_features}")
print(f"Continuous Predictors  ({len(numeric_features)}): {numeric_features}")
print(f"Omitted Candidate:         WKWN (Weeks worked - dropped for parsimony)")
print(f"Target Feature:            HIGH_EARNER (Binary: >= $90,000.00)")
print("=" * 65)

## 3. Stratified Train/Test Partitioning (80/20)
We partition the sample using an 80/20 ratio ($N_{train} = 24,000$, $N_{test} = 6,000$) stratified on `y` (`HIGH_EARNER`) with a fixed seed (`random_state=42`) to ensure identical class distributions[cite: 8].

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

print("=" * 65)
print("STRATIFIED DATA SPLIT SUMMARY")
print("=" * 65)
print(f"Training Partition (80%): {len(X_train):,} observations")
print(f"Testing Partition  (20%): {len(X_test):,} observations")
print("-" * 65)
print(f"Train High-Earner Proportion: {y_train.mean()*100:.2f}% (Class 1: {y_train.sum():,})")
print(f"Test High-Earner Proportion:  {y_test.mean()*100:.2f}% (Class 1: {y_test.sum():,})")
print("=" * 65)

## 4. Leakage-Free Preprocessing Pipeline (`ColumnTransformer`)
To guard strictly against data leakage, `StandardScaler` and `OneHotEncoder` are **fitted exclusively on `X_train`**[cite: 8]. The test split is only transformed through the fitted pipeline[cite: 8].

In [ ]:
# Construct leakage-free ColumnTransformer pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        (
            "cat",
            OneHotEncoder(drop="first", sparse_output=False, handle_unknown="ignore"),
            categorical_features,
        ),
    ]
)

# Fit exclusively on X_train, then transform both splits
X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

# Extract post-transformation feature column names
encoded_cat_names = preprocessor.named_transformers_["cat"].get_feature_names_out(
    categorical_features
)
all_feature_names = numeric_features + list(encoded_cat_names)

# Reconstruct DataFrames with split indices preserved
X_train_df = pd.DataFrame(
    X_train_encoded, columns=all_feature_names, index=X_train.index
)
X_test_df = pd.DataFrame(
    X_test_encoded, columns=all_feature_names, index=X_test.index
)

print("=" * 65)
print("ENCODED FEATURE MATRIX VERIFICATION")
print("=" * 65)
print(f"Total Model Features: {len(all_feature_names)} columns (Matches Report Section 4.4)")
print(f"X_train_df Matrix:    {X_train_df.shape}")
print(f"X_test_df Matrix:     {X_test_df.shape}")
print("-" * 65)
print("Engineered Columns:")
for idx, col in enumerate(all_feature_names, start=1):
    print(f"  {idx:02d}. {col}")
print("=" * 65)

## 5. Test Metadata Preservation and Artifact Export
We isolate unencoded demographic identifiers for the held-out test cohort to allow disaggregated subgroup auditing (RQ1 and RQ2), and export all split matrices to `data/processed/`[cite: 8].

In [ ]:
# Retain unencoded slice metadata for RQ disparity auditing
test_metadata = X_test[["SEX_LABEL", "COW_GROUP", "SCHL_TIER", "AGEP"]].copy()
test_metadata["HIGH_EARNER_ACTUAL"] = y_test

# Export processed matrices and metadata to data/processed/
X_train_df.to_csv(PROCESSED_DATA_DIR / "X_train.csv", index=False)
X_test_df.to_csv(PROCESSED_DATA_DIR / "X_test.csv", index=False)
y_train.to_csv(PROCESSED_DATA_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DATA_DIR / "y_test.csv", index=False)
test_metadata.to_csv(PROCESSED_DATA_DIR / "test_metadata.csv", index=False)

print("=" * 65)
print("[SUCCESS] All split matrices and metadata exported to data/processed/:")
print(f"  -> {PROCESSED_DATA_DIR / 'X_train.csv'}")
print(f"  -> {PROCESSED_DATA_DIR / 'X_test.csv'}")
print(f"  -> {PROCESSED_DATA_DIR / 'y_train.csv'}")
print(f"  -> {PROCESSED_DATA_DIR / 'y_test.csv'}")
print(f"  -> {PROCESSED_DATA_DIR / 'test_metadata.csv'}")
print("=" * 65)

# Verify exported files
exported_files = ["X_train.csv", "X_test.csv", "y_train.csv", "y_test.csv", "test_metadata.csv"]
all_present = all((PROCESSED_DATA_DIR / f).exists() for f in exported_files)
print(f"Export Verification: {'PASS - All 5 artifacts confirmed' if all_present else 'FAIL'}")